# Data Integration

## Import Libraries

In [1]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.4 MB/s eta 0:00:00


In [2]:
import math
import re
import json
import random
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import sentencepiece as spm
from datasets import load_dataset

import time
import torch.nn.functional as F
from dataclasses import dataclass
from sacrebleu.metrics import BLEU

## CONFIGURATION

In [3]:
class Config:
    """Central config — change these to control the whole pipeline."""

    # Language settings
    LANGUAGES      = ["ar", "en", "fr"]
    LANG_TOKENS    = {"ar": "<2ar>", "en": "<2en>", "fr": "<2fr>"}


    # Tokenizer
    VOCAB_SIZE     = 32000          # shared BPE vocabulary across all languages
    MODEL_PREFIX   = "translation_spm"   # SentencePiece model filename prefix

    # Data
    MAX_SEQ_LEN    = 64            # max tokens per sentence (longer = more VRAM)
    MIN_SEQ_LEN    = 3              # skip very short sentences
    MAX_LEN_RATIO  = 3.0            # skip pairs where len(src)/len(tgt) > ratio

    # Training split (TRAIN_RATIO is implied: 1 - VAL - TEST)
    VAL_RATIO      = 0.10
    TEST_RATIO     = 0.10

    # DataLoader
    BATCH_SIZE     = 32             # reduce to 16 if you get OOM on T4
    NUM_WORKERS    = 2

    # Paths
    DATA_DIR       = Path("data")
    SPM_DIR        = Path("tokenizer")
    CACHE_DIR      = Path("cache")

CFG = Config()

In [4]:
class huggingfaceDownloader:

    def __init__(self):
        pass

    def download_from_huggingface(
        self,
        dataset_name: str,
        config: str,
        split: str = "train",
        max_samples: Optional[int] = None
    ) -> List[Tuple[str, str, str, str]]:
        """General downloader — used for opus-100 ar-en and en-fr."""

        print(f"Downloading {dataset_name} [{config}] (streaming) ...")
        dataset = load_dataset(
            dataset_name, config, split=split,
            streaming=True
        )

        src_lang, tgt_lang = config.split("-")
        pairs = []

        for i, item in enumerate(dataset):
            if max_samples is not None and i >= max_samples:
                break
            translation = item.get("translation", {})
            src_text = translation.get(src_lang, "").strip()
            tgt_text = translation.get(tgt_lang, "").strip()
            if src_text and tgt_text:
                pairs.append((src_lang, tgt_lang, src_text, tgt_text))

        print(f"  Loaded {len(pairs):,} pairs for {config}")
        return pairs

    def download_ar_fr(
        self,
        max_samples: Optional[int] = None
    ) -> List[Tuple[str, str, str, str]]:
        """
        Download Arabic-French pairs.
        Primary:  Helsinki-NLP/opus-100  "ar-fr"  (same source as ar-en / en-fr)
        Fallback: Helsinki-NLP/un_pc     "ar-fr"  (UN parallel corpus, Parquet-native)
        Both are standard Parquet datasets — no loading script, no trust_remote_code.
        """
        # ── Primary: opus-100 ar-fr ──────────────────────────────────────
        print("Downloading ar-fr from Helsinki-NLP/opus-100 ...")
        try:
            dataset = load_dataset(
                "Helsinki-NLP/opus-100",
                "ar-fr",
                split="train",
            )
            pairs = []
            for i, item in enumerate(dataset):
                if max_samples is not None and i >= max_samples:
                    break
                translation = item.get("translation", {})
                src_text = translation.get("ar", "").strip()
                tgt_text = translation.get("fr", "").strip()
                if src_text and tgt_text:
                    pairs.append(("ar", "fr", src_text, tgt_text))

            print(f"  Loaded {len(pairs):,} ar-fr pairs from opus-100")
            return pairs

        except Exception as e:
            print(f"  opus-100 ar-fr failed: {e}")
            print("  Trying Helsinki-NLP/un_pc ar-fr (UN parallel corpus) ...")

        # ── Fallback: UN Parallel Corpus ar-fr ──────────────────────────
        try:
            dataset = load_dataset(
                "Helsinki-NLP/un_pc",
                "ar-fr",
                split="train",
            )
            pairs = []
            for i, item in enumerate(dataset):
                if max_samples is not None and i >= max_samples:
                    break
                translation = item.get("translation", {})
                src_text = translation.get("ar", "").strip()
                tgt_text = translation.get("fr", "").strip()
                if src_text and tgt_text:
                    pairs.append(("ar", "fr", src_text, tgt_text))

            print(f"  Loaded {len(pairs):,} ar-fr pairs from un_pc")
            return pairs

        except Exception as e:
            print(f"  un_pc ar-fr also failed: {e}")
            print("  ar-fr will be missing from training data.")
            return []

## DATA CLEANER

In [ ]:
import unicodedata

class DataCleaner:

    def __init__(self, config: Config = CFG):
        self.cfg = config

        # ── URL / e-mail patterns ──────────────────────────────────────────
        self._url_re   = re.compile(r"https?://\S+|www\.\S+")
        self._email_re = re.compile(r"\S+@\S+\.\S+")

        # ── Repeated-character noise  e.g. "hellooooo" → "helloo" ──────────
        self._repeat_re = re.compile(r"(.)\1{2,}")

        # ── Non-printable / control characters ────────────────────────────
        self._ctrl_re = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

        # ── Parenthesised / bracketed translator notes ─────────────────────
        # e.g. "(see note 3)", "[NB: …]", "{sic}"
        self._bracket_note_re = re.compile(
            r"[\(\[\{][^\)\]\}]{0,60}[\)\]\}]"
        )

        # ── Noise markers common in parallel corpora ───────────────────────
        # e.g. "CHAPTER 1 —", "§ 42.", numbered list markers "1."
        self._corpus_marker_re = re.compile(
            r"^(chapter|section|article|§)\s*[\d\w]+[.\s]*",
            re.IGNORECASE,
        )

        # ── French-specific: curly / angled quotes → straight ──────────────
        self._fr_quote_re = re.compile(r"[«»""„‟]")

        # ── Arabic-specific diacritic (tashkeel) strip ─────────────────────
        # Unicode range 0x064B–0x065F covers all standard Arabic harakat
        self._ar_tashkeel_re = re.compile(r"[\u064b-\u065f\u0670]")

        # ── Arabic Tatweel (kashida) ───────────────────────────────────────
        self._ar_tatweel_re  = re.compile(r"\u0640")

        # ── Arabic letter normalisation map ───────────────────────────────
        # Alef variants → bare Alef ; Teh marbuta → Heh ; Alef maqsura → Yeh
        self._ar_norm = str.maketrans(
            "إأآٱ\u0671ةى",
            "اااا" + "ا" + "هي",
        )

        # ── English-specific: smart/directional quotes → straight ──────────
        self._en_quote_re = re.compile(r"[''""‛‟]")

    # ──────────────────────────────────────────────────────────────────────
    # PHASE 1 — Light preprocessing (surface-level normalization)
    # Used before tokenizer training corpus preparation and again inside full sentence cleaning.
    # ──────────────────────────────────────────────────────────────────────
    def light_preprocess(self, text: str, lang: str) -> str:
        """
        Fast, reversible surface-level fixes that improve tokeniser coverage
        without altering meaning:

        1. Unicode NFC normalisation
        2. Strip control / non-printable characters
        3. Remove URLs and e-mail addresses (not translatable content)
        4. Collapse whitespace (tabs, newlines, multiple spaces → single space)
        5. Strip leading / trailing whitespace
        6. Remove HTML / XML tags (defensive — downloader may pass raw HTML)
        7. Normalise quotation marks per language
        8. Collapse repeated characters (noise: "wowwww" → "woww")
        """
        # 1. Unicode NFC
        text = unicodedata.normalize("NFC", text)

        # 2. Control characters
        text = self._ctrl_re.sub("", text)

        # 3. URLs / e-mails
        text = self._url_re.sub("", text)
        text = self._email_re.sub("", text)

        # 4-5. Whitespace
        text = re.sub(r"\s+", " ", text).strip()

        # 6. HTML / XML tags
        text = re.sub(r"<[^>]+>", "", text)

        # 7. Language-aware quote normalisation
        if lang == "fr":
            text = self._fr_quote_re.sub('"', text)
        elif lang == "en":
            text = self._en_quote_re.sub("'", text)

        # 8. Repeated-character collapse  ("loooool" → "lool")
        text = self._repeat_re.sub(r"\1\1", text)

        # Final whitespace trim after all substitutions
        text = text.strip()
        return text

    # ──────────────────────────────────────────────────────────────────────
    # PHASE 2 — Task-specific cleaning (translation-quality focused)
    # Applied after light_preprocess inside the full cleaning pipeline.
    # ──────────────────────────────────────────────────────────────────────
    def task_specific_clean(self, text: str, lang: str) -> str:
        """
        Deeper, task-aware cleaning tailored to MT:

        Arabic
          • Strip full tashkeel (short vowel diacritics) — unseen at inference
          • Remove tatweel / kashida decorative elongation
          • Normalise Alef/Hamza variants, Teh marbuta, Alef maqsura

        English
          • Remove corpus section / chapter markers
          • Rely on earlier quote normalization from light_preprocess

        French
          • Remove bracketed translator notes
          • Strip corpus section / chapter markers
          • Assumes quote normalization was already handled in light_preprocess
        """
        if lang == "ar":
            # Tashkeel
            text = self._ar_tashkeel_re.sub("", text)
            # Tatweel
            text = self._ar_tatweel_re.sub("", text)
            # Letter normalisation
            text = text.translate(self._ar_norm)

        elif lang == "en":
            # Smart quotes already handled; remove corpus section markers
            text = self._corpus_marker_re.sub("", text).strip()

        elif lang == "fr":
            # Remove bracketed translator notes and corpus markers
            text = self._bracket_note_re.sub("", text)
            text = self._corpus_marker_re.sub("", text).strip()

        # Re-collapse any whitespace introduced by the removals above
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # ──────────────────────────────────────────────────────────────────────
    # Unified full-cleaning entry-point: light preprocessing + task-specific cleaning
    # ──────────────────────────────────────────────────────────────────────
    def clean_text(self, text: str, lang: str) -> str:
        """
        Full preprocessing pipeline for a single sentence:
          Phase 1 → light_preprocess  (surface normalisation)
          Phase 2 → task_specific_clean  (MT-quality cleaning)
        """
        text = self.light_preprocess(text, lang)
        text = self.task_specific_clean(text, lang)
        return text

    def is_valid_pair(
        self,
        src: str,
        tgt: str,
        src_tokens: List[str],
        tgt_tokens: List[str]
    ) -> bool:
        """
        Returns False for pairs that would hurt training.
        """
        src_len = len(src_tokens)
        tgt_len = len(tgt_tokens)

        # Too short
        if src_len < self.cfg.MIN_SEQ_LEN or tgt_len < self.cfg.MIN_SEQ_LEN:
            return False

        # Too long — accounts for special tokens added during dataset construction:
        #   encoder: +1 (lang_tok) +1 (EOS) = raw + 2
        #   decoder: +1 (BOS or EOS)        = raw + 1
        # Use the stricter budget (encoder side: MAX_SEQ_LEN - 2)
        effective_max = self.cfg.MAX_SEQ_LEN - 2
        if src_len > effective_max or tgt_len > effective_max:
            return False

        # Extreme length ratio (likely a bad alignment)
        ratio = max(src_len, tgt_len) / (min(src_len, tgt_len) + 1e-6)
        if ratio > self.cfg.MAX_LEN_RATIO:
            return False

        # Source and target are identical (copy, not translation)
        if src.strip() == tgt.strip():
            return False

        # Contains only numbers/punctuation (no real content)
        if re.match(r"^[\d\s\W]+$", src) or re.match(r"^[\d\s\W]+$", tgt):
            return False

        return True

    def clean_pairs(
        self,
        pairs: List[Tuple[str, str, str, str]],
        tokenizer_fn=None
    ) -> List[Tuple[str, str, str, str]]:
        """
        Clean and filter a list of (src_lang, tgt_lang, src_text, tgt_text).
        Applies light_preprocess → task_specific_clean on every sentence, then
        filters with is_valid_pair if a tokenizer is provided.
        """
        cleaned = []
        skipped = 0

        for src_lang, tgt_lang, src, tgt in pairs:
            # Phase 1 + Phase 2 via unified clean_text entry-point
            src = self.clean_text(src, src_lang)
            tgt = self.clean_text(tgt, tgt_lang)

            # Skip empty sentences that result from aggressive cleaning
            if not src or not tgt:
                skipped += 1
                continue

            if tokenizer_fn:
                src_tokens = tokenizer_fn(src)
                tgt_tokens = tokenizer_fn(tgt)
                if not self.is_valid_pair(src, tgt, src_tokens, tgt_tokens):
                    skipped += 1
                    continue

            cleaned.append((src_lang, tgt_lang, src, tgt))

        print(f"  Cleaned: {len(cleaned):,} kept, {skipped:,} filtered out")
        return cleaned

## SENTENCEPIECE TOKENIZER TRAINER + WRAPPER

In [ ]:
class TranslationTokenizer:
    """
    Trains and wraps a SentencePiece BPE tokenizer.
    One shared vocabulary for all languages — Arabic, English, French.
    """

    def __init__(self, config: Config = CFG):
        self.cfg = config
        self.cfg.SPM_DIR.mkdir(parents=True, exist_ok=True)
        self.model_path = self.cfg.SPM_DIR / f"{self.cfg.MODEL_PREFIX}.model"
        self.sp = None

        # only lang tokens here, NOT pad/bos/eos/unk
        # pad/bos/eos/unk are registered via dedicated SPM arguments below
        self.lang_tokens = list(self.cfg.LANG_TOKENS.values())  # ["<2ar>","<2en>","<2fr>"]

    def train(self, pairs: List[Tuple[str, str, str, str]]):
        train_file = self.cfg.SPM_DIR / "spm_train_corpus.txt"
        print(f"Writing training corpus ({len(pairs)*2:,} sentences)...")

        with open(train_file, "w", encoding="utf-8") as f:
            for _, _, src, tgt in pairs:
                f.write(src + "\n")
                f.write(tgt + "\n")

        # only lang tokens in user_defined_symbols
        lang_symbols = ",".join(self.lang_tokens)   # "<2ar>,<2en>,<2fr>"

        print(f"Training SentencePiece BPE (vocab_size={self.cfg.VOCAB_SIZE})...")
        spm.SentencePieceTrainer.train(
            input                  = str(train_file),
            model_prefix           = str(self.cfg.SPM_DIR / self.cfg.MODEL_PREFIX),
            vocab_size             = self.cfg.VOCAB_SIZE,
            model_type             = "bpe",
            character_coverage     = 0.9995,
            # built-in slots for special tokens, no overlap with user_defined
            pad_id                 = 0,
            bos_id                 = 1,
            eos_id                 = 2,
            unk_id                 = 3,
            pad_piece              = "<pad>",
            bos_piece              = "<bos>",
            eos_piece              = "<eos>",
            unk_piece              = "<unk>",
            user_defined_symbols   = lang_symbols,  # IDs 4, 5, 6 — no duplicates
            shuffle_input_sentence = True,
            num_threads            = 4,
        )

        self.load()
        print(f"Tokenizer saved to {self.model_path}")
        train_file.unlink()

    def _check_loaded(self):

      """Raise if the tokenizer hasn't been loaded yet."""
      if self.sp is None:
          raise RuntimeError(
              "Tokenizer not loaded. Call .train() or .load() first."
          )

    def load(self):
        """Load a previously trained tokenizer from disk."""
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(str(self.model_path))
        print(f"Tokenizer loaded — vocab size: {self.sp.get_piece_size()}")

    def encode(self, text: str, add_bos=False, add_eos=True) -> List[int]:
        """Text → list of token IDs."""
        self._check_loaded()
        ids = self.sp.encode(text, out_type=int)
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        return ids

    def decode(self, ids: List[int]) -> str:
      """List of token IDs → clean text, all special tokens removed."""
      self._check_loaded()
      lang_ids = {self.lang_token_id(l) for l in self.cfg.LANGUAGES}
      special_ids = {
        self.pad_id,   # 0 — padding
        self.bos_id,   # 1 — beginning of sequence
        self.eos_id,   # 2 — end of sequence
        self.unk_id,   # 3 — unknown 
        *lang_ids,     # 4,5,6 — <2ar>,<2en>,<2fr> 
        }
      ids = [i for i in ids if i not in special_ids]
      return self.sp.decode(ids)

    def tokenize(self, text: str) -> List[str]:
        """Text → list of subword strings (for inspection/filtering)."""
        self._check_loaded()
        return self.sp.encode(text, out_type=str)

    def lang_token_id(self, lang: str) -> int:
        """Get the token ID for a language direction token like <2en>."""
        self._check_loaded()
        return self.sp.piece_to_id(self.cfg.LANG_TOKENS[lang])

    @property
    def vocab_size(self) -> int:
        self._check_loaded()
        return self.sp.get_piece_size()

    # hardcoded IDs, guaranteed by the trainer arguments above
    # no piece_to_id calls, no placeholder workaround, no fragile lookups
    @property
    def pad_id(self) -> int:
        self._check_loaded()
        return self.sp.pad_id()

    @property
    def bos_id(self) -> int:
        self._check_loaded()
        return self.sp.bos_id()

    @property
    def eos_id(self) -> int:
        self._check_loaded()
        return self.sp.eos_id()

    @property
    def unk_id(self) -> int:
        self._check_loaded()
        return self.sp.unk_id()

    def exists(self) -> bool:
        return self.model_path.exists()

## PYTORCH DATASET

In [ ]:
class TranslationDataset(Dataset):
    """
    PyTorch Dataset for many-to-many translation.

    Each item is a (src_lang, tgt_lang, src_text, tgt_text) tuple.
    The language direction token is prepended to the encoder input.

    Encoder input:  [<2en>] [src tokens] [<eos>]
    Decoder input:  [<bos>] [tgt tokens]           ← teacher forcing input
    Decoder target: [tgt tokens] [<eos>]            ← what we compare loss against
    """

    def __init__(
        self,
        pairs: List[Tuple[str, str, str, str]],
        tokenizer: TranslationTokenizer,
        config: Config = CFG,
        augment: bool = False
    ):
        self.pairs = pairs
        self.tok = tokenizer
        self.cfg = config
        self.augment = augment  # set True for training split only

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        src_lang, tgt_lang, src_text, tgt_text = self.pairs[idx]

        # Optionally randomly swap direction for augmentation
        # (makes model learn bidirectional translation more evenly)
        if self.augment and random.random() < 0.1:
            src_lang, tgt_lang = tgt_lang, src_lang
            src_text, tgt_text = tgt_text, src_text

        # --- Encoder input ---
        # Format: <2en> token1 token2 ... <eos>
        lang_tok_id = self.tok.lang_token_id(tgt_lang)
        src_ids = self.tok.encode(src_text, add_bos=False, add_eos=True)
        encoder_input = [lang_tok_id] + src_ids
        encoder_input = encoder_input[:self.cfg.MAX_SEQ_LEN]  # truncate if needed

        # --- Decoder input (teacher forcing) ---
        # Format: <bos> token1 token2 ...
        tgt_ids = self.tok.encode(tgt_text, add_bos=True, add_eos=False)
        tgt_ids = tgt_ids[:self.cfg.MAX_SEQ_LEN]

        # --- Decoder target (what we measure loss against) ---
        # Format: token1 token2 ... <eos>
        tgt_ids_shifted = self.tok.encode(tgt_text, add_bos=False, add_eos=True)
        tgt_ids_shifted = tgt_ids_shifted[:self.cfg.MAX_SEQ_LEN]

        return {
            "encoder_input": torch.tensor(encoder_input, dtype=torch.long),
            "decoder_input": torch.tensor(tgt_ids,         dtype=torch.long),
            "decoder_target": torch.tensor(tgt_ids_shifted, dtype=torch.long),
            "src_lang": src_lang,
            "tgt_lang": tgt_lang,
        }

## COLLATE FUNCTION (handles padding inside each batch)

In [9]:
def collate_fn(batch: List[Dict], pad_id: int) -> Dict[str, torch.Tensor]:
    """
    Pads all sequences in a batch to the same length.
    Called automatically by DataLoader.
    """
    encoder_inputs  = [item["encoder_input"]  for item in batch]
    decoder_inputs  = [item["decoder_input"]  for item in batch]
    decoder_targets = [item["decoder_target"] for item in batch]

    # Pad to longest sequence in this batch
    encoder_padded  = pad_sequence(encoder_inputs,  batch_first=True, padding_value=pad_id)
    decoder_padded  = pad_sequence(decoder_inputs,  batch_first=True, padding_value=pad_id)
    target_padded   = pad_sequence(decoder_targets, batch_first=True, padding_value=pad_id)

    return {
        "encoder_input":  encoder_padded,    # (batch, src_len)
        "decoder_input":  decoder_padded,    # (batch, tgt_len)
        "decoder_target": target_padded,     # (batch, tgt_len)
        "src_langs":      [item["src_lang"] for item in batch],
        "tgt_langs":      [item["tgt_lang"] for item in batch],
    }

## DATASET SPLITTER

In [10]:
def split_dataset(
    pairs: List[Tuple],
    config: Config = CFG,
    seed: int = 42
) -> Tuple[List, List, List]:
    """
    Shuffle and split pairs into train / val / test sets.
    Stratify by language pair so each split has balanced directions.
    """
    random.seed(seed)

    # Group by language pair for stratified splitting
    groups: Dict[str, List] = {}
    for pair in pairs:
        key = f"{pair[0]}-{pair[1]}"
        groups.setdefault(key, []).append(pair)

    train, val, test = [], [], []

    for key, group_pairs in groups.items():
        random.shuffle(group_pairs)
        n = len(group_pairs)
        n_val  = max(1, int(n * config.VAL_RATIO))
        n_test = max(1, int(n * config.TEST_RATIO))
        n_train = n - n_val - n_test

        # Safety: if group is too small, put everything in train
        if n_train < 1:
            print(f"  {key}: group too small ({n} pairs), all go to train")
            train += group_pairs
            continue

        train += group_pairs[:n_train]
        val   += group_pairs[n_train:n_train + n_val]
        test  += group_pairs[n_train + n_val:]

        print(f"  {key}: {n_train:,} train | {n_val:,} val | {n_test:,} test")

    random.shuffle(train)
    return train, val, test

## DATALOADER BUILDER

In [11]:
def build_dataloaders(
    train_pairs: List[Tuple],
    val_pairs:   List[Tuple],
    test_pairs:  List[Tuple],
    tokenizer:   TranslationTokenizer,
    config:      Config = CFG
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Build train / val / test DataLoaders."""

    from functools import partial
    _collate = partial(collate_fn, pad_id=tokenizer.pad_id)

    train_ds = TranslationDataset(train_pairs, tokenizer, config, augment=True)
    val_ds   = TranslationDataset(val_pairs,   tokenizer, config, augment=False)
    test_ds  = TranslationDataset(test_pairs,  tokenizer, config, augment=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        collate_fn=_collate,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        collate_fn=_collate,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        collate_fn=_collate,
        pin_memory=True,
    )

    print(f"\nDataLoaders ready:")
    print(f"  train: {len(train_loader):,} batches ({len(train_ds):,} samples)")
    print(f"  val:   {len(val_loader):,}  batches ({len(val_ds):,}  samples)")
    print(f"  test:  {len(test_loader):,}  batches ({len(test_ds):,}  samples)")

    return train_loader, val_loader, test_loader

## SAVE / LOAD PROCESSED DATA

In [12]:
def save_pairs(pairs: List[Tuple], filepath: Path):
    """Save processed pairs to JSON — survives Colab reconnects via Drive."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    data = [{"src_lang": s, "tgt_lang": t, "src": src, "tgt": tgt}
            for s, t, src, tgt in pairs]
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(pairs):,} pairs to {filepath}")

In [13]:
def load_pairs(filepath: Path) -> List[Tuple]:
    """Load pairs from a previously saved JSON file."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    pairs = [(d["src_lang"], d["tgt_lang"], d["src"], d["tgt"]) for d in data]
    print(f"Loaded {len(pairs):,} pairs from {filepath}")
    return pairs

## INSPECTION UTILITY

In [14]:
def inspect_batch(batch: Dict, tokenizer: TranslationTokenizer, n: int = 2):
    """
    Print n examples from a batch in human-readable form.
    Use this to sanity-check your data pipeline before training.
    """
    print("\n" + "="*60)
    print("BATCH INSPECTION")
    print("="*60)

    enc_in  = batch["encoder_input"]
    dec_in  = batch["decoder_input"]
    dec_tgt = batch["decoder_target"]

    for i in range(min(n, enc_in.size(0))):
        src_lang = batch["src_langs"][i]
        tgt_lang = batch["tgt_langs"][i]

        enc_ids = enc_in[i].tolist()
        dec_ids = dec_in[i].tolist()
        tgt_ids = dec_tgt[i].tolist()

        # Remove padding for display
        pad = tokenizer.pad_id
        enc_ids = [x for x in enc_ids if x != pad]
        dec_ids = [x for x in dec_ids if x != pad]
        tgt_ids = [x for x in tgt_ids if x != pad]

        print(f"\nExample {i+1} | {src_lang} → {tgt_lang}")
        print(f"  encoder input ids : {enc_ids[:12]}...")
        print(f"  encoder input text: {tokenizer.decode(enc_ids[1:])}")  # skip lang token
        print(f"  decoder input ids : {dec_ids[:12]}...")
        print(f"  decoder target    : {tokenizer.decode(tgt_ids)}")
        print(f"  enc shape: {enc_in[i].shape} | dec shape: {dec_in[i].shape}")

    print(f"\nBatch shapes:")
    print(f"  encoder_input  : {batch['encoder_input'].shape}")
    print(f"  decoder_input  : {batch['decoder_input'].shape}")
    print(f"  decoder_target : {batch['decoder_target'].shape}")

    print("="*60 + "\n")

# Build Model

## POSITIONAL ENCODING

In [15]:
class PositionalEncoding(nn.Module):
    """
    Adds position information to token embeddings using fixed sinusoids.
    Registered as a buffer — moves to GPU with .to(device), not a parameter.
    """

    def __init__(self, d_model: int, max_seq_len: int = 128, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe       = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_seq_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])

In [16]:
class TranslationTransformer(nn.Module):
    """
    Many-to-many translation model built on top of nn.Transformer.
    """

    def __init__(
        self,
        vocab_size:  int,
        d_model:     int   = 256,
        n_heads:     int   = 4,
        n_layers:    int   = 4,
        d_ff:        int   = 1024,
        dropout:     float = 0.1,
        max_seq_len: int   = 64,
        pad_id:      int   = 0,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_id  = pad_id
        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc   = PositionalEncoding(d_model, max_seq_len, dropout)

        self.transformer = nn.Transformer(
            d_model            = d_model,
            nhead              = n_heads,
            num_encoder_layers = n_layers,
            num_decoder_layers = n_layers,
            dim_feedforward    = d_ff,
            dropout            = dropout,
            batch_first        = True,
        )

        self.projection = nn.Linear(d_model, vocab_size, bias=False)
        self.projection.weight = self.embedding.weight

        self._init_weights()

        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model ready — {total:,} trainable parameters")

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src: torch.Tensor) -> torch.Tensor:
        return src == self.pad_id

    def make_tgt_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        tgt_len = tgt.size(1)
        return torch.triu(
            torch.ones((tgt_len, tgt_len), device=tgt.device, dtype=torch.bool),
            diagonal=1
        )

    def make_tgt_padding_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        return tgt == self.pad_id

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
    ) -> torch.Tensor:
        src_key_padding_mask = self.make_src_mask(src)
        tgt_mask             = self.make_tgt_mask(tgt)
        tgt_key_padding_mask = self.make_tgt_padding_mask(tgt)

        src_emb = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))

        out = self.transformer(
            src                     = src_emb,
            tgt                     = tgt_emb,
            tgt_mask                = tgt_mask,
            src_key_padding_mask    = src_key_padding_mask,
            tgt_key_padding_mask    = tgt_key_padding_mask,
            memory_key_padding_mask = src_key_padding_mask,
        )
        return self.projection(out)

    def encode(self, src: torch.Tensor) -> tuple:
        src_key_padding_mask = self.make_src_mask(src)
        src_emb = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        memory  = self.transformer.encoder(
            src_emb,
            src_key_padding_mask=src_key_padding_mask
        )
        return memory, src_key_padding_mask

    def decode(
        self,
        tgt:                  torch.Tensor,
        memory:               torch.Tensor,
        src_key_padding_mask: torch.Tensor,
    ) -> torch.Tensor:
        tgt_mask             = self.make_tgt_mask(tgt)
        tgt_key_padding_mask = self.make_tgt_padding_mask(tgt)
        tgt_emb = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))

        out = self.transformer.decoder(
            tgt                     = tgt_emb,
            memory                  = memory,
            tgt_mask                = tgt_mask,
            tgt_key_padding_mask    = tgt_key_padding_mask,
            memory_key_padding_mask = src_key_padding_mask,
        )
        return self.projection(out)

## BUILD FUNCTION

In [17]:
def build_transformer(
    vocab_size:  int,
    max_seq_len=CFG.MAX_SEQ_LEN,
    pad_id:      int   = 0,
    d_model:     int   = 256,
    n_heads:     int   = 4,
    n_layers:    int   = 4,
    d_ff:        int   = 1024,
    dropout:     float = 0.1
) -> TranslationTransformer:

    return TranslationTransformer(
        vocab_size  = vocab_size,
        d_model     = d_model,
        n_heads     = n_heads,
        n_layers    = n_layers,
        d_ff        = d_ff,
        dropout     = dropout,
        max_seq_len = max_seq_len,
        pad_id      = pad_id,
    )

In [18]:
@dataclass
class TrainCFG:
    # Paths
    checkpoint_dir: Path = Path("checkpoints")
    best_model_path: Path = Path("checkpoints/best_model.pt")

    # Training
    n_epochs:       int   = 15
    warmup_steps:   int   = 4000
    max_grad_norm:  float = 1.0     # gradient clipping threshold

    # Loss
    label_smoothing: float = 0.1   # smooth targets away from hard 0/1

    # Early stopping
    patience:       int   = 4      # stop if val loss doesn't improve for this many epochs

    # Mixed precision (free speedup on T4)
    use_amp:        bool  = True

    # Logging
    log_every_n_steps: int = 100

TCFG = TrainCFG()

In [19]:
class LabelSmoothedCrossEntropy(nn.Module):
    def __init__(self, vocab_size: int, pad_id: int, smoothing: float = 0.1):
        super().__init__()

        if not (0.0 <= smoothing < 1.0):
            raise ValueError("smoothing must be in [0, 1).")
        if vocab_size <= 2:
            raise ValueError("vocab_size must be > 2 when excluding target and PAD.")

        self.vocab_size = vocab_size
        self.pad_id = pad_id
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        B, L, V = logits.shape

        if V != self.vocab_size:
            raise ValueError(f"logits vocab size {V} != expected {self.vocab_size}")
        if not (0 <= self.pad_id < V):
            raise ValueError("pad_id is out of range")

        logits = logits.reshape(-1, V)
        targets = targets.reshape(-1)

        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.smoothing / (V - 2))
            smooth_targets[:, self.pad_id] = 0.0
            smooth_targets.scatter_(1, targets.unsqueeze(1), self.confidence)

            pad_rows = targets.eq(self.pad_id).unsqueeze(1)
            smooth_targets.masked_fill_(pad_rows, 0.0)

        log_probs = F.log_softmax(logits, dim=-1)
        loss = -(smooth_targets * log_probs).sum(dim=-1)

        non_pad_mask = targets.ne(self.pad_id)
        loss = loss.masked_select(non_pad_mask)

        if loss.numel() == 0:
            return logits.new_tensor(0.0)

        return loss.mean()

In [20]:
@torch.no_grad()
def beam_decode(
    model,
    src: torch.Tensor,
    tokenizer: TranslationTokenizer,
    device: torch.device,
    max_len: int | None = None,
    beam_size: int = 4,
    length_penalty: float = 0.6,
) -> list:
    """
    Beam search decoding.
    Uses model.max_seq_len automatically if max_len is not provided.
    """
    model.eval()
    memory, src_mask = model.encode(src)

    if max_len is None:
        max_len = model.max_seq_len

    # Defined once outside the loop — fixes NameError risk
    def normalized_score(b):
        length = b["seq"].size(0)
        return b["score"] / (length ** length_penalty)

    init_seq  = torch.tensor([tokenizer.bos_id], device=device)
    beams     = [{"seq": init_seq, "score": 0.0, "done": False}]
    completed = []

    for _ in range(max_len):
        # Only expand beams that are still active
        active_beams = [b for b in beams if not b["done"]]
        if not active_beams:
            break

        candidates = []

        for b in active_beams:
            tgt       = b["seq"].unsqueeze(0)
            logits    = model.decode(tgt, memory, src_mask)
            logits    = logits[0, -1, :]
            log_probs = torch.log_softmax(logits, dim=-1)

            topk_log_probs, topk_ids = log_probs.topk(beam_size)

            for log_p, tok_id in zip(topk_log_probs, topk_ids):
                new_score = b["score"] + log_p.item()
                new_seq   = torch.cat([b["seq"], tok_id.unsqueeze(0)])
                done      = tok_id.item() == tokenizer.eos_id

                candidates.append({
                    "seq":   new_seq,
                    "score": new_score,
                    "done":  done,
                })

        candidates.sort(key=normalized_score, reverse=True)

        beams = []
        for c in candidates:
            if c["done"]:
                completed.append(c)
            else:
                beams.append(c)
            if len(beams) == beam_size:
                break

        if len(completed) >= beam_size:
            break

    final_pool = completed if completed else beams
    best = max(final_pool, key=normalized_score)

    ids = best["seq"][1:].tolist()       # drop leading <bos>
    if ids and ids[-1] == tokenizer.eos_id:
        ids = ids[:-1]                   # drop trailing <eos>

    return ids

In [21]:
@torch.no_grad()
def greedy_decode_batch(
    model,
    src: torch.Tensor,          # (batch, src_len) — full batch at once
    tokenizer: TranslationTokenizer,
    device: torch.device,
    max_len: int | None = None,
) -> list[list[int]]:
    """
    Greedy decoding for a full batch in parallel.
    ~10-20x faster than sample-by-sample beam search.
    Used for BLEU tracking during training.
    """
    model.eval()
    batch_size = src.size(0)
    if max_len is None:
        max_len = model.max_seq_len

    memory, src_mask = model.encode(src)                    # encode whole batch once

    # Start every sequence with <bos>
    ys = torch.full(
        (batch_size, 1), tokenizer.bos_id,
        dtype=torch.long, device=device
    )
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(max_len):
        logits = model.decode(ys, memory, src_mask)         # (batch, cur_len, vocab)
        next_token = logits[:, -1, :].argmax(dim=-1)        # (batch,) greedy pick
        ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)
        finished |= next_token.eq(tokenizer.eos_id)
        if finished.all():
            break

    # Convert each row to a list of IDs, stripping <bos> and everything after <eos>
    results = []
    for i in range(batch_size):
        ids = ys[i, 1:].tolist()                            # drop leading <bos>
        if tokenizer.eos_id in ids:
            ids = ids[:ids.index(tokenizer.eos_id)]         # drop <eos> and beyond
        results.append(ids)

    return results

In [ ]:
@torch.no_grad()
def evaluate_bleu(
    model,
    loader,
    tokenizer,
    device: torch.device,
    max_batches: int = 10,
) -> dict:
    model.eval()
    bleu_metric = BLEU(effective_order=True)

    hypotheses = []
    references  = []

    for i, batch in enumerate(loader):
        if i >= max_batches:
            break

        src     = batch["encoder_input"].to(device)
        tgt_ids = batch["decoder_target"].to(device)

        # Decode the entire batch at once — no inner loop over samples
        all_pred_ids = greedy_decode_batch(model, src, tokenizer, device)

        for j, pred_ids in enumerate(all_pred_ids):
            pred_str = tokenizer.decode(pred_ids)

            ref_ids = tgt_ids[j].tolist()
            ref_ids = [t for t in ref_ids
                       if t not in (tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id)]
            ref_str = tokenizer.decode(ref_ids)

            hypotheses.append(pred_str)
            references.append(ref_str)

    score = bleu_metric.corpus_score(hypotheses, [references])
    return {"bleu": round(score.score, 2)}

In [23]:
@torch.no_grad()
def evaluate_bleu_beam(
    model,
    loader,
    tokenizer,
    device: torch.device,
    max_batches: int = 50,
    beam_size: int = 4,
) -> dict:
    """
    BLEU evaluation using beam search decoding.
    Slower but more accurate — use for final test evaluation only.
    """
    model.eval()
    bleu_metric = BLEU(effective_order=True)

    hypotheses = []
    references  = []

    for i, batch in enumerate(loader):
        if i >= max_batches:
            break

        src     = batch["encoder_input"].to(device)
        tgt_ids = batch["decoder_target"].to(device)

        for j in range(src.size(0)):
            pred_ids = beam_decode(
                model, src[j].unsqueeze(0), tokenizer, device,
                beam_size=beam_size,
            )
            pred_str = tokenizer.decode(pred_ids)

            ref_ids = tgt_ids[j].tolist()
            ref_ids = [t for t in ref_ids
                       if t not in (tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id)]
            ref_str = tokenizer.decode(ref_ids)

            hypotheses.append(pred_str)
            references.append(ref_str)

    score = bleu_metric.corpus_score(hypotheses, [references])
    return {"bleu": round(score.score, 2)}

##  ONE TRAINING EPOCH

In [24]:
def train_one_epoch(
    model,
    loader,
    criterion,
    scheduler,
    device: torch.device,
    scaler,
    epoch: int,
) -> float:
    model.train()
    total_loss   = 0.0
    total_tokens = 0
    step_loss    = 0.0
    t0           = time.time()

    for step, batch in enumerate(loader):
        src     = batch["encoder_input"].to(device)
        tgt_in  = batch["decoder_input"].to(device)
        tgt_out = batch["decoder_target"].to(device)

        # 1. Clear gradients from the previous step FIRST
        scheduler.optimizer.zero_grad()

        with torch.amp.autocast(device.type, enabled=TCFG.use_amp):
            logits = model(src, tgt_in)
            loss   = criterion(logits, tgt_out)

        # 2. Scale and backpropagate
        scaler.scale(loss).backward()

        # 3. Unscale before clipping so clip threshold is in real gradient units
        scaler.unscale_(scheduler.optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), TCFG.max_grad_norm)

        # 4. Apply the optimizer step (updates weights)
        scaler.step(scheduler.optimizer)
        scaler.update()

        # 5. Advance the Noam LR schedule AFTER the optimizer step
        scheduler.step()

        n_tokens      = tgt_out.ne(criterion.pad_id).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens
        step_loss    += loss.item()

        if (step + 1) % TCFG.log_every_n_steps == 0:
            elapsed = time.time() - t0
            avg     = step_loss / TCFG.log_every_n_steps
            lr      = scheduler._rate
            print(
                f"  Epoch {epoch:02d} | Step {step+1:5d} | "
                f"Loss {avg:.4f} | LR {lr:.2e} | {elapsed:.1f}s"
            )
            step_loss = 0.0
            t0 = time.time()

    return total_loss / max(total_tokens, 1)

 ## ONE VALIDATION EPOCH

In [25]:
@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device: torch.device,
) -> float:
    model.eval()
    total_loss   = 0.0
    total_tokens = 0

    for batch in loader:
        src     = batch["encoder_input"].to(device)
        tgt_in  = batch["decoder_input"].to(device)
        tgt_out = batch["decoder_target"].to(device)

        with torch.amp.autocast(device.type, enabled=TCFG.use_amp):
            logits = model(src, tgt_in)
            loss   = criterion(logits, tgt_out)

        n_tokens      = tgt_out.ne(criterion.pad_id).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens

    return total_loss / max(total_tokens, 1)

##  CHECKPOINT HELPERS

In [26]:
def save_checkpoint(model, scheduler, epoch: int, val_loss: float, path: Path):
    torch.save({
        "epoch":      epoch,
        "model":      model.state_dict(),
        "optimizer":  scheduler.optimizer.state_dict(),
        "val_loss":   val_loss,
        "step":       scheduler._step,
    }, path)
    print(f"  Checkpoint saved → {path}")

In [27]:
def load_checkpoint(model, scheduler, path: Path):
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(ckpt["model"])
    scheduler.optimizer.load_state_dict(ckpt["optimizer"])
    scheduler._step = ckpt["step"]
    scheduler._rate = scheduler._compute_rate()  # restore LR state for consistent resume
    print(f"  Loaded checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")
    return ckpt["epoch"], ckpt["val_loss"]

In [28]:
class NoamScheduler:
    """LR = d_model^-0.5 * min(step^-0.5, step * warmup^-1.5)"""
    def __init__(self, optimizer, d_model: int, warmup_steps: int):
        self.optimizer    = optimizer
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        self._step        = 0
        self._rate        = 0.0

    def step(self):
        self._step += 1
        rate = self._compute_rate()
        for p in self.optimizer.param_groups:
            p["lr"] = rate
        self._rate = rate


    def _compute_rate(self) -> float:
        s, w, d = self._step, self.warmup_steps, self.d_model
        if s == 0:
            return 0.0
        return (d ** -0.5) * min(s ** -0.5, s * (w ** -1.5))

In [ ]:
def train(
    model,
    train_loader,
    val_loader,
    tokenizer,
    device: torch.device,
    resume_from: Optional[Path] = None,
    bleu_every_n_epochs: int = 5,
):
    TCFG.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    criterion = LabelSmoothedCrossEntropy(
        vocab_size = tokenizer.vocab_size,
        pad_id     = tokenizer.pad_id,
        smoothing  = TCFG.label_smoothing,
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9,
    )
    scheduler = NoamScheduler(
        optimizer, d_model=model.d_model, warmup_steps=TCFG.warmup_steps,
    )
    scaler = torch.amp.GradScaler(device.type, enabled=TCFG.use_amp)

    start_epoch      = 1
    best_val_loss    = float("inf")
    patience_counter = 0

    if resume_from and resume_from.exists():
        start_epoch, best_val_loss = load_checkpoint(model, scheduler, resume_from)
        start_epoch += 1

    print(f"\nStarting training on {device}")
    print(f"Epochs: {TCFG.n_epochs} | Warmup: {TCFG.warmup_steps} steps | "
          f"Label smoothing: {TCFG.label_smoothing}\n")

    history = {"train_loss": [], "val_loss": [], "bleu": []}

    for epoch in range(start_epoch, TCFG.n_epochs + 1):
        print(f"{'─'*60}")
        print(f"Epoch {epoch}/{TCFG.n_epochs}")

        train_loss = train_one_epoch(
            model, train_loader, criterion, scheduler, device, scaler, epoch
        )
        val_loss = validate(model, val_loader, criterion, device)

        # Only run expensive beam-search BLEU every N epochs
        if epoch % bleu_every_n_epochs == 0 or epoch == TCFG.n_epochs:
            bleu_results = evaluate_bleu(model, val_loader, tokenizer, device)
            bleu = bleu_results["bleu"]
            print(f"  BLEU evaluated: {bleu:.1f}")
        else:
            bleu = history["bleu"][-1] if history["bleu"] else 0.0  # carry last known value

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["bleu"].append(bleu)

        print(
            f"  train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_ppl={math.exp(min(val_loss, 100)):.1f} | "
            f"BLEU={bleu:.1f}"
        )

        save_checkpoint(
            model, scheduler, epoch, val_loss,
            TCFG.checkpoint_dir / f"epoch_{epoch:02d}.pt"
        )

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            save_checkpoint(model, scheduler, epoch, val_loss, TCFG.best_model_path)
            print(f"  New best model! val_loss={val_loss:.4f}")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{TCFG.patience})")

        if patience_counter >= TCFG.patience:
            print(f"\nEarly stopping triggered after epoch {epoch}.")
            break

    print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")
    print(f"Best model saved at: {TCFG.best_model_path}")
    return history

In [30]:
# ─────────────────────────────────────────────
#  ARTIFACT EXPORT
#  Saves model config, training history, and
#  run summary as JSON for reproducibility.
# ─────────────────────────────────────────────

ARTIFACT_DIR = Path("artifacts")

def export_artifacts(
    model,
    tokenizer: TranslationTokenizer,
    history: dict,
    test_bleu: dict,
    device: torch.device,
    total_epochs_run: int,
):
    """
    Export three JSON artifacts after training:
      artifacts/model_config.json      — architecture + training hyperparams
      artifacts/training_history.json  — per-epoch losses and BLEU
      artifacts/run_summary.json       — final metrics, output paths, device info
    """
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    # ── 1) model_config.json ──────────────────────────────────────────
    model_config = {
        "architecture": {
            "vocab_size":  tokenizer.vocab_size,
            "d_model":     model.d_model,
            "n_heads":     model.transformer.nhead,
            "n_encoder_layers": len(model.transformer.encoder.layers),
            "n_decoder_layers": len(model.transformer.decoder.layers),
            "d_ff":        model.transformer.encoder.layers[0].linear1.out_features,
            "max_seq_len": CFG.MAX_SEQ_LEN,
            "pad_id":      model.pad_id,
        },
        "training": {
            "n_epochs":        TCFG.n_epochs,
            "warmup_steps":    TCFG.warmup_steps,
            "label_smoothing": TCFG.label_smoothing,
            "max_grad_norm":   TCFG.max_grad_norm,
            "batch_size":      CFG.BATCH_SIZE,
            "use_amp":         TCFG.use_amp,
            "patience":        TCFG.patience,
        },
        "data": {
            "languages":     CFG.LANGUAGES,
            "lang_tokens":   CFG.LANG_TOKENS,
            "max_seq_len":   CFG.MAX_SEQ_LEN,
            "min_seq_len":   CFG.MIN_SEQ_LEN,
            "max_len_ratio": CFG.MAX_LEN_RATIO,
        },
        "tokenizer": {
            "type":       "sentencepiece_bpe",
            "vocab_size": tokenizer.vocab_size,
            "pad_id":     tokenizer.pad_id,
            "bos_id":     tokenizer.bos_id,
            "eos_id":     tokenizer.eos_id,
            "unk_id":     tokenizer.unk_id,
        },
    }
    config_path = ARTIFACT_DIR / "model_config.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(model_config, f, indent=2, ensure_ascii=False)
    print(f"  Saved → {config_path}")

    # ── 2) training_history.json ──────────────────────────────────────
    history_path = ARTIFACT_DIR / "training_history.json"
    with open(history_path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    print(f"  Saved → {history_path}")

    # ── 3) run_summary.json ───────────────────────────────────────────
    best_epoch_idx = history["val_loss"].index(min(history["val_loss"]))
    run_summary = {
        "device":           str(device),
        "total_parameters": sum(p.numel() for p in model.parameters()),
        "epochs_run":       total_epochs_run,
        "best_epoch":       best_epoch_idx + 1,
        "best_val_loss":    history["val_loss"][best_epoch_idx],
        "best_val_bleu":    history["bleu"][best_epoch_idx],
        "test_bleu":        test_bleu["bleu"],
        "output_files": {
            "best_model":    str(TCFG.best_model_path),
            "tokenizer_model": str(CFG.SPM_DIR / f"{CFG.MODEL_PREFIX}.model"),
            "tokenizer_vocab": str(CFG.SPM_DIR / f"{CFG.MODEL_PREFIX}.vocab"),
            "model_config":    str(ARTIFACT_DIR / "model_config.json"),
            "training_history": str(ARTIFACT_DIR / "training_history.json"),
            "run_summary":     str(ARTIFACT_DIR / "run_summary.json"),
        },
    }
    summary_path = ARTIFACT_DIR / "run_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(run_summary, f, indent=2, ensure_ascii=False)
    print(f"  Saved → {summary_path}")

## ENTRY POINT — STAGED PIPELINE

Run the next cells one by one. Each cell handles a single stage

In [31]:
# ─────────────────────────────────────────────
#  PIPELINE STAGE HELPERS
#  Run the cells below one by one.
# ─────────────────────────────────────────────

PIPELINE_STATE = {}

def stage_prepare_environment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CFG.DATA_DIR.mkdir(parents=True, exist_ok=True)
    CFG.CACHE_DIR.mkdir(parents=True, exist_ok=True)
    TCFG.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    PIPELINE_STATE["device"] = device
    print(f"Device: {device}")
    print(f"Data dir: {CFG.DATA_DIR}")
    print(f"Cache dir: {CFG.CACHE_DIR}")
    print(f"Checkpoint dir: {TCFG.checkpoint_dir}")
    return device


def stage_load_raw_pairs(max_samples_per_pair, force_redownload: bool = False):
    cache_file = CFG.CACHE_DIR / "all_pairs.json"

    if cache_file.exists() and not force_redownload:
        print("Cache found — loading pairs (skip re-download)")
        all_pairs = load_pairs(cache_file)
    else:
        downloader = huggingfaceDownloader()
        all_pairs = []
        _half = lambda n: n // 2 if n is not None else None

        ar_en = downloader.download_from_huggingface(
            "Helsinki-NLP/opus-100", "ar-en", max_samples=max_samples_per_pair
        )
        all_pairs += ar_en
        en_ar = [(t, s, tgt, src) for s, t, src, tgt in ar_en[:_half(max_samples_per_pair)]]
        all_pairs += en_ar

        ar_fr = downloader.download_ar_fr(max_samples=max_samples_per_pair)
        all_pairs += ar_fr
        if ar_fr:
            fr_ar = [(t, s, tgt, src) for s, t, src, tgt in ar_fr[:_half(max_samples_per_pair)]]
            all_pairs += fr_ar

        try:
            en_fr = downloader.download_from_huggingface(
                "Helsinki-NLP/opus-100", "en-fr", max_samples=max_samples_per_pair
            )
            all_pairs += en_fr
            fr_en = [(t, s, tgt, src) for s, t, src, tgt in en_fr[:_half(max_samples_per_pair)]]
            all_pairs += fr_en
        except Exception as e:
            print(f"  en-fr not available, skipping: {e}")

        print(f"\nTotal raw pairs: {len(all_pairs):,}")
        save_pairs(all_pairs, cache_file)

    PIPELINE_STATE["all_pairs"] = all_pairs
    PIPELINE_STATE["max_samples_per_pair"] = max_samples_per_pair
    print(f"Loaded raw pairs: {len(all_pairs):,}")
    return all_pairs

def stage_light_preprocess_pairs():
    all_pairs = PIPELINE_STATE["all_pairs"]
    cleaner   = DataCleaner(CFG)

    print(f"Light-preprocessing {len(all_pairs):,} pairs before tokenizer training...")
    preprocessed = []
    skipped = 0
    for src_lang, tgt_lang, src, tgt in all_pairs:
        src = cleaner.light_preprocess(src, src_lang)
        tgt = cleaner.light_preprocess(tgt, tgt_lang)
        if src and tgt:
            preprocessed.append((src_lang, tgt_lang, src, tgt))
        else:
            skipped += 1

    print(f"  Preprocessed: {len(preprocessed):,} kept, {skipped:,} dropped (empty after cleaning)")
    PIPELINE_STATE["preprocessed_pairs"] = preprocessed
    return preprocessed


def stage_prepare_tokenizer(retrain_tokenizer: bool = False):
    preprocessed_pairs = PIPELINE_STATE["preprocessed_pairs"]
    tokenizer = TranslationTokenizer(CFG)

    if tokenizer.exists() and not retrain_tokenizer:
        print("Tokenizer found — loading (skip retraining)")
        tokenizer.load()
    else:
        print("Training tokenizer...")
        tokenizer.train(preprocessed_pairs)

    PIPELINE_STATE["tokenizer"] = tokenizer
    return tokenizer


def stage_clean_pairs():
    all_pairs = PIPELINE_STATE["all_pairs"]
    tokenizer = PIPELINE_STATE["tokenizer"]

    print("Cleaning pairs...")
    cleaner = DataCleaner(CFG)
    clean_pairs = cleaner.clean_pairs(all_pairs, tokenizer_fn=tokenizer.tokenize)
    PIPELINE_STATE["clean_pairs"] = clean_pairs
    print(f"Clean pairs: {len(clean_pairs):,}")
    return clean_pairs


def stage_split_pairs():
    clean_pairs = PIPELINE_STATE["clean_pairs"]
    print("Splitting dataset...")
    train_pairs, val_pairs, test_pairs = split_dataset(clean_pairs, CFG)
    PIPELINE_STATE["train_pairs"] = train_pairs
    PIPELINE_STATE["val_pairs"] = val_pairs
    PIPELINE_STATE["test_pairs"] = test_pairs
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,} | Test: {len(test_pairs):,}")
    return train_pairs, val_pairs, test_pairs


def stage_build_dataloaders_and_inspect():
    tokenizer = PIPELINE_STATE["tokenizer"]
    train_pairs = PIPELINE_STATE["train_pairs"]
    val_pairs = PIPELINE_STATE["val_pairs"]
    test_pairs = PIPELINE_STATE["test_pairs"]

    print("Building DataLoaders...")
    train_loader, val_loader, test_loader = build_dataloaders(
        train_pairs, val_pairs, test_pairs, tokenizer, CFG
    )

    PIPELINE_STATE["train_loader"] = train_loader
    PIPELINE_STATE["val_loader"] = val_loader
    PIPELINE_STATE["test_loader"] = test_loader

    print("Running sanity check on first batch...")
    first_batch = next(iter(train_loader))
    inspect_batch(first_batch, tokenizer, n=2)
    return train_loader, val_loader, test_loader


def stage_build_model(resume_from: Optional[Path] = None):
    tokenizer = PIPELINE_STATE["tokenizer"]
    device = PIPELINE_STATE["device"]

    model = build_transformer(
        vocab_size=tokenizer.vocab_size,
        pad_id=tokenizer.pad_id,
        max_seq_len=CFG.MAX_SEQ_LEN,
    ).to(device)

    PIPELINE_STATE["model"] = model
    PIPELINE_STATE["resume_from"] = resume_from
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    if resume_from:
        print(f"Resume checkpoint requested: {resume_from}")
    return model


def stage_train_model(resume_from: Optional[Path] = None):
    model = PIPELINE_STATE["model"]
    train_loader = PIPELINE_STATE["train_loader"]
    val_loader = PIPELINE_STATE["val_loader"]
    tokenizer = PIPELINE_STATE["tokenizer"]
    device = PIPELINE_STATE["device"]

    history = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        tokenizer=tokenizer,
        device=device,
        resume_from=resume_from,
    )
    PIPELINE_STATE["history"] = history
    return history


def stage_evaluate_best_checkpoint(checkpoint_path: Optional[Path] = None):
    device      = PIPELINE_STATE["device"]
    model       = PIPELINE_STATE["model"]
    tokenizer   = PIPELINE_STATE["tokenizer"]
    test_loader = PIPELINE_STATE["test_loader"]

    checkpoint_path = checkpoint_path or TCFG.best_model_path
    print(f"Loading checkpoint: {checkpoint_path}")
    best_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(best_ckpt["model"])

    print("Running final test evaluation with beam search (beam_size=4)...")
    test_bleu = evaluate_bleu_beam(model, test_loader, tokenizer, device)

    PIPELINE_STATE["test_bleu"] = test_bleu
    print(f"Test BLEU (beam search): {test_bleu['bleu']:.2f}")
    return test_bleu


def stage_export_run_artifacts():
    model = PIPELINE_STATE["model"]
    tokenizer = PIPELINE_STATE["tokenizer"]
    history = PIPELINE_STATE["history"]
    test_bleu = PIPELINE_STATE["test_bleu"]
    device = PIPELINE_STATE["device"]
    total_epochs_run = len(history["train_loss"])

    export_artifacts(model, tokenizer, history, test_bleu, device, total_epochs_run)

    print("\nAll output files")
    print("=" * 60)
    print(f"  {TCFG.best_model_path}")
    print(f"  {CFG.SPM_DIR / f'{CFG.MODEL_PREFIX}.model'}")
    print(f"  {CFG.SPM_DIR / f'{CFG.MODEL_PREFIX}.vocab'}")
    print(f"  {ARTIFACT_DIR / 'model_config.json'}")
    print(f"  {ARTIFACT_DIR / 'training_history.json'}")
    print(f"  {ARTIFACT_DIR / 'run_summary.json'}")
    print("=" * 60)

## Run Pipeline

In [32]:
# Stage 0 — runtime options
MAX_SAMPLES_PER_PAIR = 15000
RETRAIN_TOKENIZER = False
FORCE_REDOWNLOAD = False
RESUME_CHECKPOINT = None  # e.g. TCFG.checkpoint_dir / "epoch_03.pt"


In [33]:
# Stage 1 — environment
device = stage_prepare_environment()

Device: cuda
Data dir: data
Cache dir: cache
Checkpoint dir: checkpoints


In [34]:
# Stage 2 — raw data download / cache load
all_pairs = stage_load_raw_pairs(
    max_samples_per_pair=MAX_SAMPLES_PER_PAIR,
    force_redownload=FORCE_REDOWNLOAD,
)
print(f"Sample raw pair: {all_pairs[0] if all_pairs else None}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

  Loaded 15,000 pairs for ar-en


ar-fr/test-00000-of-00001.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

...

ar-fr/train-00016-of-00018.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

ar-fr/train-00017-of-00018.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20281645 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

  Loaded 15,000 ar-fr pairs from un_pc
  Loaded 15,000 pairs for en-fr

Total raw pairs: 67,500
Saved 67,500 pairs to cache/all_pairs.json
Loaded raw pairs: 67,500
Sample raw pair: ('ar', 'en', 'و هذه؟', 'And this?')


In [35]:
# Stage 2b — light preprocessing
preprocessed_pairs = stage_light_preprocess_pairs()
print(f"Sample preprocessed pair: {preprocessed_pairs[0] if preprocessed_pairs else None}")

Light-preprocessing 67,500 pairs before tokenizer training...
  Preprocessed: 67,457 kept, 43 dropped (empty after cleaning)
Sample preprocessed pair: ('ar', 'en', 'و هذه؟', 'And this?')


In [36]:
# Stage 3 — tokenizer
tokenizer = stage_prepare_tokenizer(retrain_tokenizer=RETRAIN_TOKENIZER)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

Training tokenizer...
Writing training corpus (134,914 sentences)...
Training SentencePiece BPE (vocab_size=32000)...
Tokenizer loaded — vocab size: 32000
Tokenizer saved to tokenizer/translation_spm.model
Tokenizer vocab size: 32000


In [37]:
# Stage 4 — cleaning
clean_pairs = stage_clean_pairs()
print(f"Sample clean pair: {clean_pairs[0] if clean_pairs else None}")

Cleaning pairs...
  Cleaned: 57,311 kept, 10,189 filtered out
Clean pairs: 57,311
Sample clean pair: ('ar', 'en', 'و هذه؟', 'And this?')


In [38]:
# Stage 5 — split
train_pairs, val_pairs, test_pairs = stage_split_pairs()

Splitting dataset...
  ar-en: 10,658 train | 1,332 val | 1,332 test
  en-ar: 5,224 train | 652 val | 652 test
  ar-fr: 9,674 train | 1,209 val | 1,209 test
  fr-ar: 4,848 train | 606 val | 606 test
  en-fr: 10,317 train | 1,289 val | 1,289 test
  fr-en: 5,132 train | 641 val | 641 test
Train: 45,853 | Val: 5,729 | Test: 5,729


In [39]:
# Stage 6 — dataloaders + sanity check
train_loader, val_loader, test_loader = stage_build_dataloaders_and_inspect()
print(f"Batches | train={len(train_loader)} | val={len(val_loader)} | test={len(test_loader)}")

Building DataLoaders...

DataLoaders ready:
  train: 1,433 batches (45,853 samples)
  val:   180  batches (5,729  samples)
  test:  180  batches (5,729  samples)
Running sanity check on first batch...

BATCH INSPECTION

Example 1 | en → fr
  encoder input ids : [6, 254, 11, 31902, 31850, 29072, 2924, 109, 1360, 13795, 2]...
  encoder input text: Won't Father talk to him?"
  decoder input ids : [1, 6947, 31899, 114, 200, 1076, 123, 465, 70, 31902, 17905, 391]...
  decoder target    : Est-ce que mon Père n'irait pas lui parler? "
  enc shape: torch.Size([53]) | dec shape: torch.Size([52])

Example 2 | ar → fr
  encoder input ids : [6, 1326, 11749, 31907, 1661, 49, 122, 12541, 31907, 1772, 288, 3716]...
  encoder input text: عام ٧٦٩١، بمافيها القدس، وعلي السكان العرب
  decoder input ids : [1, 6638, 31878, 146, 2762, 12216, 31878, 13691, 3309, 12137, 75, 265]...
  decoder target    : palestinien, y compris Jérusalem, occupé depuis 1967, et sur la
  enc shape: torch.Size([53]) | dec shape: 

In [40]:
# Stage 7 — build model
model = stage_build_model(resume_from=RESUME_CHECKPOINT)


Model ready — 15,565,824 trainable parameters
Model parameters: 15,565,824


In [41]:
# Stage 8 — training
history = stage_train_model(resume_from=RESUME_CHECKPOINT)
history


Starting training on cuda
Epochs: 15 | Warmup: 4000 steps | Label smoothing: 0.1

────────────────────────────────────────────────────────────
Epoch 1/15
  Epoch 01 | Step   100 | Loss 10.2307 | LR 2.47e-05 | 7.3s
  Epoch 01 | Step   200 | Loss 9.7273 | LR 4.94e-05 | 5.9s
  Epoch 01 | Step   300 | Loss 8.9578 | LR 7.41e-05 | 5.4s
  Epoch 01 | Step   400 | Loss 8.2735 | LR 9.88e-05 | 6.1s
  Epoch 01 | Step   500 | Loss 7.9714 | LR 1.24e-04 | 5.4s
  Epoch 01 | Step   600 | Loss 7.6557 | LR 1.48e-04 | 6.0s
  Epoch 01 | Step   700 | Loss 7.3576 | LR 1.73e-04 | 5.5s
  Epoch 01 | Step   800 | Loss 7.1416 | LR 1.98e-04 | 6.0s
  Epoch 01 | Step   900 | Loss 6.9440 | LR 2.22e-04 | 5.4s
  Epoch 01 | Step  1000 | Loss 6.8116 | LR 2.47e-04 | 6.7s
  Epoch 01 | Step  1100 | Loss 6.7092 | LR 2.72e-04 | 5.4s
  Epoch 01 | Step  1200 | Loss 6.5911 | LR 2.96e-04 | 5.7s
  Epoch 01 | Step  1300 | Loss 6.4653 | LR 3.21e-04 | 5.7s
  Epoch 01 | Step  1400 | Loss 6.3864 | LR 3.46e-04 | 5.4s
  train_loss=7.631

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


  BLEU evaluated: 1.0
  train_loss=5.1091 | val_loss=5.0261 | val_ppl=152.3 | BLEU=1.0
  Checkpoint saved → checkpoints/epoch_05.pt
  Checkpoint saved → checkpoints/best_model.pt
  New best model! val_loss=5.0261
────────────────────────────────────────────────────────────
Epoch 6/15
  Epoch 06 | Step   100 | Loss 4.8949 | LR 7.33e-04 | 6.2s
  Epoch 06 | Step   200 | Loss 4.9670 | LR 7.28e-04 | 5.5s
  Epoch 06 | Step   300 | Loss 4.9349 | LR 7.23e-04 | 6.2s
  Epoch 06 | Step   400 | Loss 4.8920 | LR 7.19e-04 | 5.5s
  Epoch 06 | Step   500 | Loss 5.0248 | LR 7.14e-04 | 6.2s
  Epoch 06 | Step   600 | Loss 4.9525 | LR 7.09e-04 | 5.5s
  Epoch 06 | Step   700 | Loss 5.0139 | LR 7.05e-04 | 6.0s
  Epoch 06 | Step   800 | Loss 5.0099 | LR 7.00e-04 | 5.5s
  Epoch 06 | Step   900 | Loss 4.9462 | LR 6.96e-04 | 6.1s
  Epoch 06 | Step  1000 | Loss 4.9626 | LR 6.92e-04 | 5.5s
  Epoch 06 | Step  1100 | Loss 4.9098 | LR 6.87e-04 | 5.6s
  Epoch 06 | Step  1200 | Loss 5.0200 | LR 6.83e-04 | 5.8s
  Epoch

{'train_loss': [7.631009472769304,
  5.926673172114973,
  5.533912405646618,
  5.303416950598238,
  5.1090728433728625,
  4.959645921137806,
  4.836588061695373,
  4.73417407541394,
  4.6461683048687314,
  4.573507324564188,
  4.504713140889741,
  4.452789991794249,
  4.3997847940793555,
  4.3585595164657,
  4.317326246438882],
 'val_loss': [6.217464796489606,
  5.57138561808228,
  5.333272637740242,
  5.163730851474627,
  5.026110929371261,
  4.927282337523223,
  4.85659053881294,
  4.796904767388406,
  4.7593370926358425,
  4.722301584256108,
  4.695017904252727,
  4.685697297461796,
  4.6494413895446,
  4.640196730132194,
  4.622908303048728],
 'bleu': [0.0,
  0.0,
  0.0,
  0.0,
  1.01,
  1.01,
  1.01,
  1.01,
  1.01,
  2.38,
  2.38,
  2.38,
  2.38,
  2.38,
  3.72]}

In [42]:
# 1. Ensure environment is ready
stage_prepare_environment()

# 2. Load pairs and tokenizer from disk (no retraining)
stage_load_raw_pairs(MAX_SAMPLES_PER_PAIR)
stage_light_preprocess_pairs()
tokenizer = stage_prepare_tokenizer(retrain_tokenizer=False)

# 3. Rebuild the test dataloader
clean_pairs = stage_clean_pairs()
train_p, val_p, test_pairs = stage_split_pairs()
_, _, test_loader = build_dataloaders(train_p, val_p, test_pairs, tokenizer, CFG)

# Update PIPELINE_STATE manually to satisfy stage_evaluate_best_checkpoint
PIPELINE_STATE['tokenizer'] = tokenizer
PIPELINE_STATE['test_loader'] = test_loader

# 4. Rebuild Model Architecture
model = build_transformer(
    vocab_size=tokenizer.vocab_size,
    pad_id=tokenizer.pad_id,
    max_seq_len=CFG.MAX_SEQ_LEN
).to(PIPELINE_STATE['device'])
PIPELINE_STATE['model'] = model

Device: cuda
Data dir: data
Cache dir: cache
Checkpoint dir: checkpoints
Cache found — loading pairs (skip re-download)
Loaded 67,500 pairs from cache/all_pairs.json
Loaded raw pairs: 67,500
Light-preprocessing 67,500 pairs before tokenizer training...
  Preprocessed: 67,457 kept, 43 dropped (empty after cleaning)
Tokenizer found — loading (skip retraining)
Tokenizer loaded — vocab size: 32000
Cleaning pairs...
  Cleaned: 57,311 kept, 10,189 filtered out
Clean pairs: 57,311
Splitting dataset...
  ar-en: 10,658 train | 1,332 val | 1,332 test
  en-ar: 5,224 train | 652 val | 652 test
  ar-fr: 9,674 train | 1,209 val | 1,209 test
  fr-ar: 4,848 train | 606 val | 606 test
  en-fr: 10,317 train | 1,289 val | 1,289 test
  fr-en: 5,132 train | 641 val | 641 test
Train: 45,853 | Val: 5,729 | Test: 5,729

DataLoaders ready:
  train: 1,433 batches (45,853 samples)
  val:   180  batches (5,729  samples)
  test:  180  batches (5,729  samples)
Model ready — 15,565,824 trainable parameters


In [43]:
assert TCFG.best_model_path.exists(), f"Checkpoint not found: {TCFG.best_model_path}"
assert tokenizer.exists(),            f"Tokenizer not found: {CFG.SPM_DIR}"

In [44]:
# 5. Run Evaluation
test_bleu = stage_evaluate_best_checkpoint(TCFG.best_model_path)
print(f"\nFinal Test Result: {test_bleu}")

Loading checkpoint: checkpoints/best_model.pt
Running final test evaluation with beam search (beam_size=4)...
Test BLEU (beam search): 4.68

Final Test Result: {'bleu': 4.68}


In [45]:
# Stage 10 — export artifacts
stage_export_run_artifacts()

  Saved → artifacts/model_config.json
  Saved → artifacts/training_history.json
  Saved → artifacts/run_summary.json

All output files
  checkpoints/best_model.pt
  tokenizer/translation_spm.model
  tokenizer/translation_spm.vocab
  artifacts/model_config.json
  artifacts/training_history.json
  artifacts/run_summary.json


-##-